# Cox-Merz Rule Validation for Rheological Data

This notebook demonstrates the Cox-Merz rule validation transform in RheoJAX.
The Cox-Merz rule is an empirical relationship that connects oscillatory and
steady-shear rheology:

$$|\eta^*(\omega)| = \eta(\dot{\gamma}) \quad \text{at} \quad \omega = \dot{\gamma}$$

where $\eta^*(\omega) = |G^*(\omega)| / \omega$ is the complex viscosity magnitude
and $\eta(\dot{\gamma})$ is the steady-shear viscosity.

**Learning Objectives:**
- Understand the Cox-Merz rule and its physical basis
- Generate synthetic oscillation and flow curve data using Maxwell model
- Apply the `CoxMerz` transform to compare $|\eta^*(\omega)|$ vs $\eta(\dot{\gamma})$
- Identify materials that violate the Cox-Merz rule
- Explore the effect of the tolerance parameter on pass/fail classification

**Prerequisites:** Basic understanding of oscillatory and steady-shear rheology,
complex modulus $G^* = G' + iG''$

**Estimated Time:** 20-30 minutes

## Google Colab Setup

If running in Google Colab, execute this cell first to install RheoJAX:

In [ ]:
# Google Colab Setup - Run this cell first!
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q rheojax
    import os
    os.environ['JAX_ENABLE_X64'] = 'true'
    print("\u2713 RheoJAX installed successfully!")
    print("\u2713 Float64 precision enabled")

## Setup and Imports

In [ ]:
# Configure matplotlib for inline plotting
# MUST come before importing matplotlib
%matplotlib inline

import warnings

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from rheojax.core.data import RheoData
from rheojax.core.jax_config import safe_import_jax, verify_float64
from rheojax.transforms.cox_merz import CoxMerz

jax, jnp = safe_import_jax()
verify_float64()

print(f"\u2713 JAX float64 precision enabled (default dtype bits: {jax.config.jax_default_dtype_bits})")

np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
warnings.filterwarnings('ignore', message='.*non-interactive.*')

## Theory: The Cox-Merz Rule

The Cox-Merz rule (Cox & Merz, 1958) is an empirical relationship stating that
the complex viscosity magnitude from oscillatory experiments equals the
steady-shear viscosity at the same numerical value of frequency and shear rate:

$$|\eta^*(\omega)| = \eta(\dot{\gamma}) \quad \text{at} \quad \omega = \dot{\gamma}$$

The complex viscosity is computed from the complex modulus:

$$\eta^*(\omega) = \frac{G^*(\omega)}{i\omega} = \frac{G'(\omega) - iG''(\omega)}{\omega}$$

so its magnitude is:

$$|\eta^*(\omega)| = \frac{|G^*(\omega)|}{\omega} = \frac{\sqrt{G'^2 + G''^2}}{\omega}$$

**When does the Cox-Merz rule hold?**
- Linear polymer melts and solutions in the terminal regime
- Entangled polymer systems with simple chain dynamics

**When does it fail?**
- Physically or chemically cross-linked gels
- Colloidal suspensions and emulsions
- Associating polymers with strain-induced network disruption
- Materials exhibiting shear banding

Deviations from the Cox-Merz rule are diagnostic: $\eta(\dot{\gamma}) < |\eta^*(\omega)|$
typically indicates shear-induced structural breakdown not present in the linear regime.

## Generate Synthetic Data

We generate synthetic oscillation data from a 3-mode Generalized Maxwell model,
for which the Cox-Merz rule can be shown to hold exactly in the terminal limit.

In [ ]:
# 3-mode Maxwell model parameters
G_modes = np.array([1000.0, 500.0, 200.0])   # Pa — elastic moduli
tau_modes = np.array([10.0, 1.0, 0.1])        # s  — relaxation times

omega = np.logspace(-2, 2, 50)  # rad/s — angular frequency sweep

# Storage modulus G'(omega) and loss modulus G''(omega) from Maxwell model
G_prime = np.sum(
    G_modes * (omega[:, None] * tau_modes) ** 2
    / (1 + (omega[:, None] * tau_modes) ** 2),
    axis=1,
)
G_double_prime = np.sum(
    G_modes * omega[:, None] * tau_modes
    / (1 + (omega[:, None] * tau_modes) ** 2),
    axis=1,
)
G_star = G_prime + 1j * G_double_prime

osc_data = RheoData(
    x=omega,
    y=G_star,
    metadata={'test_mode': 'oscillation'},
)

print(f"Oscillation data: {len(omega)} points, omega in [{omega[0]:.2g}, {omega[-1]:.2g}] rad/s")
print(f"G' range:  [{G_prime.min():.2g}, {G_prime.max():.2g}] Pa")
print(f"G'' range: [{G_double_prime.min():.2g}, {G_double_prime.max():.2g}] Pa")

# Preview complex viscosity magnitude
eta_star_preview = np.abs(G_star) / omega
print(f"|eta*| range: [{eta_star_preview.min():.2g}, {eta_star_preview.max():.2g}] Pa.s")

In [ ]:
# Construct steady-shear flow curve that obeys Cox-Merz exactly
# |eta*(omega)| at omega = gamma_dot gives eta(gamma_dot)
eta_star = np.abs(G_star) / omega

gamma_dot = omega.copy()          # shear rate grid equals frequency grid
eta_steady = eta_star.copy()      # Cox-Merz holds exactly

# Add small Gaussian noise (~3%) for realism
eta_steady_noisy = eta_steady * (1 + 0.03 * np.random.randn(len(eta_steady)))
sigma_noisy = eta_steady_noisy * gamma_dot   # stress = viscosity * shear rate

flow_data = RheoData(
    x=gamma_dot,
    y=sigma_noisy,
    metadata={'quantity': 'stress'},
)

print(f"Flow curve data: {len(gamma_dot)} points, gamma_dot in [{gamma_dot[0]:.2g}, {gamma_dot[-1]:.2g}] s^-1")
print(f"Noise level: 3% relative (seed=42)")

## Apply Cox-Merz Validation

The `CoxMerz` transform takes the two `RheoData` objects, interpolates them
onto a common log-spaced rate grid, and computes the relative deviation
$\delta = |\eta^* - \eta| / |\eta^*|$ at each point.

In [ ]:
cox_merz = CoxMerz(tolerance=0.10, n_points=50)
result_data, meta = cox_merz._transform([osc_data, flow_data])
result = cox_merz.result

print("Cox-Merz Validation Results (Maxwell model + 3% noise):")
print(f"  Mean deviation: {result.mean_deviation:.4f} ({result.mean_deviation * 100:.1f}%)")
print(f"  Max deviation:  {result.max_deviation:.4f} ({result.max_deviation * 100:.1f}%)")
print(f"  Tolerance:      10%")
print(f"  Passes (<=10%): {'\u2713 YES' if result.passes else '\u2717 NO'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: viscosity comparison ---
ax = axes[0]
ax.loglog(
    omega,
    np.abs(G_star) / omega,
    'b-',
    linewidth=2,
    label=r'$|\eta^*(\omega)|$ (oscillation)',
)
ax.loglog(
    gamma_dot,
    eta_steady_noisy,
    'rs',
    markersize=5,
    alpha=0.7,
    label=r'$\eta(\dot{\gamma})$ (steady shear)',
)
ax.set_xlabel(r'$\omega$ (rad/s) or $\dot{\gamma}$ (s$^{-1}$)', fontsize=12)
ax.set_ylabel(r'Viscosity (Pa$\cdot$s)', fontsize=12)
ax.set_title('Cox-Merz Rule: Maxwell Model (3% noise)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)

# --- Right panel: pointwise deviation ---
ax2 = axes[1]
ax2.semilogx(
    result.common_rates,
    result.deviation * 100,
    'g-o',
    linewidth=1.5,
    markersize=4,
    label='Relative deviation',
)
ax2.axhline(
    cox_merz.tolerance * 100,
    color='r',
    linestyle='--',
    linewidth=1.5,
    label=f'Tolerance ({cox_merz.tolerance * 100:.0f}%)',
)
ax2.axhline(
    result.mean_deviation * 100,
    color='orange',
    linestyle=':',
    linewidth=1.5,
    label=f'Mean deviation ({result.mean_deviation * 100:.1f}%)',
)
ax2.set_xlabel(r'Rate (rad/s or s$^{-1}$)', fontsize=12)
ax2.set_ylabel('Relative deviation (%)', fontsize=12)
ax2.set_title('Cox-Merz Deviation Profile', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, which='both', alpha=0.3)
ax2.set_ylim(bottom=0)

pass_label = 'PASS' if result.passes else 'FAIL'
pass_color = 'green' if result.passes else 'red'
fig.suptitle(
    f'Cox-Merz Validation: {pass_label}  |  Mean deviation: {result.mean_deviation * 100:.1f}%',
    fontsize=13,
    color=pass_color,
    fontweight='bold',
)
plt.tight_layout()
plt.show()

## Case Study: Material That Violates Cox-Merz

Many complex fluids violate the Cox-Merz rule because shear flow disrupts
microstructure that is preserved under small-amplitude oscillation.
Common examples include:

- **Associating polymers**: transient network breaks down under shear
- **Entangled biopolymers** (actin, collagen): strain-induced alignment
- **Filled rubbers**: Payne effect — filler network disrupted by shear

Here we simulate a material where $\eta(\dot{\gamma})$ falls significantly below
$|\eta^*(\omega)|$ at high rates, mimicking strong shear thinning beyond the
linear viscoelastic regime.

In [ ]:
# Oscillation data: same Maxwell model as before (linear regime)
# The oscillation measurement is unaffected — small strains preserve structure.

# Flow curve: associating polymer with enhanced shear thinning
# eta(gamma_dot) = eta*(gamma_dot) * f(gamma_dot)
# where f < 1 at high rates due to network breakdown
gamma_dot_viol = omega.copy()
eta_star_ref = np.abs(G_star) / omega

# Shear-induced structural breakdown factor: sigmoid from 1.0 to 0.15
gamma_dot_crit = 1.0   # s^-1 — critical rate for network breakdown
breakdown_factor = 1.0 - 0.85 / (1.0 + np.exp(-2.0 * (np.log10(gamma_dot_viol) - np.log10(gamma_dot_crit))))
eta_viol = eta_star_ref * breakdown_factor
sigma_viol = eta_viol * gamma_dot_viol * (1 + 0.02 * np.random.randn(len(gamma_dot_viol)))

flow_data_viol = RheoData(
    x=gamma_dot_viol,
    y=sigma_viol,
    metadata={'quantity': 'stress'},
)

# Apply Cox-Merz transform
cox_merz_viol = CoxMerz(tolerance=0.10, n_points=50)
result_data_viol, _ = cox_merz_viol._transform([osc_data, flow_data_viol])
result_viol = cox_merz_viol.result

print("Cox-Merz Validation Results (Cox-Merz violating material):")
print(f"  Mean deviation: {result_viol.mean_deviation:.4f} ({result_viol.mean_deviation * 100:.1f}%)")
print(f"  Max deviation:  {result_viol.max_deviation:.4f} ({result_viol.max_deviation * 100:.1f}%)")
print(f"  Passes (<=10%): {'\u2713 YES' if result_viol.passes else '\u2717 NO'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: viscosity comparison ---
ax = axes[0]
ax.loglog(
    omega,
    np.abs(G_star) / omega,
    'b-',
    linewidth=2,
    label=r'$|\eta^*(\omega)|$ (oscillation)',
)
ax.loglog(
    gamma_dot_viol,
    eta_viol,
    'r--',
    linewidth=2,
    label=r'$\eta(\dot{\gamma})$ (steady shear, violating)',
)
ax.set_xlabel(r'$\omega$ (rad/s) or $\dot{\gamma}$ (s$^{-1}$)', fontsize=12)
ax.set_ylabel(r'Viscosity (Pa$\cdot$s)', fontsize=12)
ax.set_title('Cox-Merz Rule: Associating Polymer', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)

# Shade the deviation region
ax.fill_between(
    omega,
    eta_viol,
    np.abs(G_star) / omega,
    alpha=0.15,
    color='red',
    label='Violation region',
)

# --- Right panel: deviation profile ---
ax2 = axes[1]
ax2.semilogx(
    result_viol.common_rates,
    result_viol.deviation * 100,
    'r-o',
    linewidth=1.5,
    markersize=4,
    label='Relative deviation',
)
ax2.axhline(
    cox_merz_viol.tolerance * 100,
    color='k',
    linestyle='--',
    linewidth=1.5,
    label=f'Tolerance ({cox_merz_viol.tolerance * 100:.0f}%)',
)
ax2.axhline(
    result_viol.mean_deviation * 100,
    color='orange',
    linestyle=':',
    linewidth=1.5,
    label=f'Mean deviation ({result_viol.mean_deviation * 100:.1f}%)',
)
ax2.set_xlabel(r'Rate (rad/s or s$^{-1}$)', fontsize=12)
ax2.set_ylabel('Relative deviation (%)', fontsize=12)
ax2.set_title('Cox-Merz Deviation Profile (Failing)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, which='both', alpha=0.3)
ax2.set_ylim(bottom=0)

pass_label = 'PASS' if result_viol.passes else 'FAIL'
pass_color = 'green' if result_viol.passes else 'red'
fig.suptitle(
    f'Cox-Merz Validation: {pass_label}  |  Mean deviation: {result_viol.mean_deviation * 100:.1f}%',
    fontsize=13,
    color=pass_color,
    fontweight='bold',
)
plt.tight_layout()
plt.show()

## Effect of Tolerance Parameter

The `tolerance` parameter controls how strictly the Cox-Merz rule is applied.
A material passes if its mean relative deviation is at or below the tolerance.

Typical choices:
- `tolerance=0.05` (5%): strict, suitable for well-characterized polymer standards
- `tolerance=0.10` (10%): moderate, common default
- `tolerance=0.20` (20%): lenient, useful for screening complex fluids

Here we sweep tolerance values and show the pass/fail transition for both
the compliant Maxwell material and the violating associating polymer.

In [ ]:
tolerances = np.linspace(0.01, 0.60, 60)   # 1% to 60%

passes_maxwell = []
passes_viol = []

for tol in tolerances:
    # Maxwell (compliant)
    cm = CoxMerz(tolerance=float(tol), n_points=50)
    cm._transform([osc_data, flow_data])
    passes_maxwell.append(cm.result.passes)

    # Violating material
    cm_v = CoxMerz(tolerance=float(tol), n_points=50)
    cm_v._transform([osc_data, flow_data_viol])
    passes_viol.append(cm_v.result.passes)

# Fixed deviations (computed once)
mean_dev_maxwell = result.mean_deviation * 100
mean_dev_viol = result_viol.mean_deviation * 100

fig, ax = plt.subplots(figsize=(10, 5))

ax.fill_between(
    tolerances * 100,
    [int(p) for p in passes_maxwell],
    alpha=0.3,
    color='blue',
    label=f'Maxwell (mean dev = {mean_dev_maxwell:.1f}%) — passes above dashed blue line',
)
ax.fill_between(
    tolerances * 100,
    [int(p) for p in passes_viol],
    alpha=0.3,
    color='red',
    label=f'Associating polymer (mean dev = {mean_dev_viol:.1f}%) — passes above dashed red line',
)

ax.axvline(mean_dev_maxwell, color='blue', linestyle='--', linewidth=1.5,
           label=f'Maxwell threshold ({mean_dev_maxwell:.1f}%)')
ax.axvline(mean_dev_viol, color='red', linestyle='--', linewidth=1.5,
           label=f'Violating threshold ({mean_dev_viol:.1f}%)')

ax.set_xlabel('Tolerance (%)', fontsize=12)
ax.set_ylabel('Passes (1 = yes, 0 = no)', fontsize=12)
ax.set_title('Cox-Merz Pass/Fail vs Tolerance Setting', fontsize=12)
ax.set_yticks([0, 1])
ax.set_yticklabels(['FAIL', 'PASS'])
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Maxwell material passes at tolerance >= {mean_dev_maxwell:.1f}%")
print(f"Associating polymer passes at tolerance >= {mean_dev_viol:.1f}%")

## Key Takeaways

1. **Cox-Merz connects two measurement modes**: The rule bridges oscillatory
   (linear viscoelastic) and steady-shear (nonlinear) measurements. When it
   holds, a single oscillatory sweep can predict the full flow curve.

2. **Valid for simple polymer melts**: Linear entangled polymers with simple
   chain dynamics generally satisfy Cox-Merz within 5-15%. The Generalized
   Maxwell model obeys it exactly in the terminal regime.

3. **Failures indicate microstructural complexity**: When $\eta(\dot{\gamma}) < |\eta^*(\omega)|$,
   shear flow disrupts microstructure beyond what small oscillatory strains access.
   This is diagnostic for gels, filled systems, and associating polymers.

4. **The `CoxMerz` transform handles both input conventions**: Flow data can be
   provided as stress $\sigma(\dot{\gamma})$ (converted internally via $\eta = \sigma/\dot{\gamma}$)
   or directly as viscosity $\eta(\dot{\gamma})$ (set `metadata={'is_viscosity': True}`).

5. **Tolerance is context-dependent**: Use a stricter tolerance (5%) for precision
   polymer standards, and a more lenient value (20%) for initial screening of
   complex industrial fluids.

**Further Reading:**
- Cox, W. P. & Merz, E. H. (1958). *Correlation of dynamic and steady flow viscosities.*
  Journal of Polymer Science, 28(118), 619-622.
- Larson, R. G. (1999). *The Structure and Rheology of Complex Fluids.* Oxford University Press.

## Session Information

In [ ]:
import sys
import scipy
import rheojax

print(f"Python version:      {sys.version}")
print(f"RheoJAX version:     {rheojax.__version__}")
print(f"JAX version:         {jax.__version__}")
print(f"JAX devices:         {jax.devices()}")
print(f"NumPy version:       {np.__version__}")
print(f"SciPy version:       {scipy.__version__}")
print(f"Matplotlib version:  {plt.matplotlib.__version__}")